## Imports

In [2]:
from pprint import pprint
import pandas as pd
try:
    import pyodbc
except ModuleNotFoundError:
    ! pip install pyodbc
    import pyodbc
import os
try:
    import boto3
except ModuleNotFoundError:
    ! pip install boto3
    import boto3
import datetime as dt
from tqdm import tqdm
import numpy as np

## Constants

In [3]:
str_project = os.getcwd().split('\\')[4].replace('_', '-')
print(f'Project: {str_project}')
str_task = os.getcwd().split('\\')[5]
print(f'Task: {str_task}')
str_subtask = os.getcwd().split('\\')[6]
print(f'Subtask: {str_subtask}')
str_subtask2 = os.getcwd().split('\\')[7]
print(f'Subtask2: {str_subtask2}')
str_dirname_output = './output'

Project: 20240509-christian-internship
Task: 06_ml_in_python
Subtask: 01_home_credit_model
Subtask2: 03_preprocessing


In [4]:
try:
    os.mkdir(str_dirname_output)
except FileExistsError:
    pass

## Function

In [5]:
def upload_to_s3(aws_access_key_id, aws_secret_access_key, str_local_path, str_bucket_key, str_bucket_name):
    # init boto 3 client
    cls_client = boto3.client(
        's3',
        aws_access_key_id=aws_access_key_id,
        aws_secret_access_key=aws_secret_access_key,
        aws_session_token=None,
    )
    
    cls_client.upload_file(
        str_local_path,
        str_bucket_name,
        str_bucket_key
    )

## Read Data

In [7]:
str_filename = 'df_training.csv'
str_local_path = f'./input/{str_filename}'
df_training = pd.read_csv(str_local_path)

FileNotFoundError: [Errno 2] No such file or directory: './output/df_train.csv'

In [8]:
str_filename = 'df_validation.csv'
str_local_path = f'./input/{str_filename}'
df_validation = pd.read_csv(str_local_path)

FileNotFoundError: [Errno 2] No such file or directory: './output/df_valid.csv'

In [19]:
str_filename = 'df_testing.csv'
str_local_path = f'./input/{str_filename}'
df_testing = pd.read_csv(str_local_path)

## Clean Data

### df_training

In [27]:
missing_vals = df_training.isnull().mean() * 100
missing_cols = missing_vals[missing_vals > 0]
missing_cols = missing_cols.sort_values(ascending=False)
cols_to_remove = missing_cols[missing_cols > 65].index.tolist()
df_training_clean = df_training.drop(columns=cols_to_remove)
numerical_cols = df_training_clean.select_dtypes(include=['float64', 'int64']).columns.drop('TARGET')
categorical_cols = df_training_clean.select_dtypes(include=['object']).columns
df_training_clean[numerical_cols] = df_training_clean[numerical_cols].fillna(df_training_clean[numerical_cols].median())
for col in categorical_cols:
    df_training_clean[col] = df_training_clean[col].fillna(df_training_clean[col].mode()[0])
df_training_clean

,RowNum,SK_ID_CURR,TARGET,NAME_CONTRACT_TYPE,CODE_GENDER,FLAG_OWN_CAR,FLAG_OWN_REALTY,CNT_CHILDREN,AMT_INCOME_TOTAL,AMT_CREDIT,...,FLAG_DOCUMENT_18,FLAG_DOCUMENT_19,FLAG_DOCUMENT_20,FLAG_DOCUMENT_21,AMT_REQ_CREDIT_BUREAU_HOUR,AMT_REQ_CREDIT_BUREAU_DAY,AMT_REQ_CREDIT_BUREAU_WEEK,AMT_REQ_CREDIT_BUREAU_MON,AMT_REQ_CREDIT_BUREAU_QRT,AMT_REQ_CREDIT_BUREAU_YEAR
0,75000,314008,0,Cash loans,F,N,Y,0,112500,1237500,...,0,0,0,0,0.0,0.0,0.0,0.0,0.0,6.0
1,75001,314009,0,Cash loans,F,N,Y,0,67500,107820,...,0,0,0,0,0.0,0.0,0.0,0.0,0.0,8.0
2,75002,314012,0,Cash loans,M,N,Y,1,360000,665892,...,0,0,0,0,0.0,0.0,0.0,0.0,0.0,0.0
3,75003,314019,0,Cash loans,F,N,Y,1,72000,270000,...,0,0,0,0,0.0,0.0,1.0,0.0,0.0,3.0
4,75004,314021,0,Cash loans,F,N,Y,0,139500,298512,...,0,0,0,0,0.0,0.0,0.0,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
24995,99995,385070,0,Cash loans,F,N,N,0,180000,755190,...,0,0,0,0,0.0,0.0,0.0,0.0,0.0,3.0
24996,99996,385072,1,Cash loans,F,N,Y,0,225000,408780,...,0,0,0,0,0.0,0.0,0.0,0.0,0.0,0.0
24997,99997,385073,1,Cash loans,M,Y,Y,0,157500,450000,...,0,0,0,0,0.0,0.0,0.0,1.0,1.0,1.0
24998,99998,385074,0,Cash loans,F,N,Y,1,180000,195543,...,0,0,0,0,0.0,0.0,0.0,1.0,0.0,2.0


### df_validation

In [29]:
missing_vals = df_validation.isnull().mean() * 100
missing_cols = missing_vals[missing_vals > 0]
missing_cols = missing_cols.sort_values(ascending=False)
cols_to_remove = missing_cols[missing_cols > 65].index.tolist()
df_validation_clean = df_validation.drop(columns=cols_to_remove)
numerical_cols = df_validation_clean.select_dtypes(include=['float64', 'int64']).columns.drop('TARGET')
categorical_cols = df_validation_clean.select_dtypes(include=['object']).columns
df_validation_clean[numerical_cols] = df_validation_clean[numerical_cols].fillna(df_validation_clean[numerical_cols].median())
for col in categorical_cols:
    df_validation_clean[col] = df_validation_clean[col].fillna(df_validation_clean[col].mode()[0])
df_validation_clean

,RowNum,SK_ID_CURR,TARGET,NAME_CONTRACT_TYPE,CODE_GENDER,FLAG_OWN_CAR,FLAG_OWN_REALTY,CNT_CHILDREN,AMT_INCOME_TOTAL,AMT_CREDIT,...,FLAG_DOCUMENT_18,FLAG_DOCUMENT_19,FLAG_DOCUMENT_20,FLAG_DOCUMENT_21,AMT_REQ_CREDIT_BUREAU_HOUR,AMT_REQ_CREDIT_BUREAU_DAY,AMT_REQ_CREDIT_BUREAU_WEEK,AMT_REQ_CREDIT_BUREAU_MON,AMT_REQ_CREDIT_BUREAU_QRT,AMT_REQ_CREDIT_BUREAU_YEAR
0,75000,314008,0,Cash loans,F,N,Y,0,112500,1237500,...,0,0,0,0,0.0,0.0,0.0,0.0,0.0,6.0
1,75001,314009,0,Cash loans,F,N,Y,0,67500,107820,...,0,0,0,0,0.0,0.0,0.0,0.0,0.0,8.0
2,75002,314012,0,Cash loans,M,N,Y,1,360000,665892,...,0,0,0,0,0.0,0.0,0.0,0.0,0.0,0.0
3,75003,314019,0,Cash loans,F,N,Y,1,72000,270000,...,0,0,0,0,0.0,0.0,1.0,0.0,0.0,3.0
4,75004,314021,0,Cash loans,F,N,Y,0,139500,298512,...,0,0,0,0,0.0,0.0,0.0,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
24995,99995,385070,0,Cash loans,F,N,N,0,180000,755190,...,0,0,0,0,0.0,0.0,0.0,0.0,0.0,3.0
24996,99996,385072,1,Cash loans,F,N,Y,0,225000,408780,...,0,0,0,0,0.0,0.0,0.0,0.0,0.0,0.0
24997,99997,385073,1,Cash loans,M,Y,Y,0,157500,450000,...,0,0,0,0,0.0,0.0,0.0,1.0,1.0,1.0
24998,99998,385074,0,Cash loans,F,N,Y,1,180000,195543,...,0,0,0,0,0.0,0.0,0.0,1.0,0.0,2.0


### df_testing

In [30]:
missing_vals = df_testing.isnull().mean() * 100
missing_cols = missing_vals[missing_vals > 0]
missing_cols = missing_cols.sort_values(ascending=False)
cols_to_remove = missing_cols[missing_cols > 65].index.tolist()
df_testing_clean = df_testing.drop(columns=cols_to_remove)
numerical_cols = df_testing_clean.select_dtypes(include=['float64', 'int64']).columns.drop('TARGET')
categorical_cols = df_testing_clean.select_dtypes(include=['object']).columns
df_testing_clean[numerical_cols] = df_testing_clean[numerical_cols].fillna(df_testing_clean[numerical_cols].median())
for col in categorical_cols:
    df_testing_clean[col] = df_testing_clean[col].fillna(df_testing_clean[col].mode()[0])
df_testing_clean

,RowNum,SK_ID_CURR,TARGET,NAME_CONTRACT_TYPE,CODE_GENDER,FLAG_OWN_CAR,FLAG_OWN_REALTY,CNT_CHILDREN,AMT_INCOME_TOTAL,AMT_CREDIT,...,FLAG_DOCUMENT_18,FLAG_DOCUMENT_19,FLAG_DOCUMENT_20,FLAG_DOCUMENT_21,AMT_REQ_CREDIT_BUREAU_HOUR,AMT_REQ_CREDIT_BUREAU_DAY,AMT_REQ_CREDIT_BUREAU_WEEK,AMT_REQ_CREDIT_BUREAU_MON,AMT_REQ_CREDIT_BUREAU_QRT,AMT_REQ_CREDIT_BUREAU_YEAR
0,100000,385079,0,Cash loans,M,Y,Y,2,135000,251091,...,0,0,0,0,0.0,0.0,0.0,0.0,0.0,1.0
1,100001,385080,0,Cash loans,F,N,Y,0,112500,495000,...,0,0,0,0,0.0,0.0,0.0,0.0,0.0,1.0
2,100002,385083,0,Cash loans,M,N,Y,0,270000,579942,...,0,0,0,0,0.0,0.0,0.0,2.0,0.0,5.0
3,100003,385084,0,Revolving loans,F,N,Y,0,229500,202500,...,0,0,0,0,0.0,0.0,0.0,0.0,0.0,6.0
4,100004,385085,0,Cash loans,M,N,Y,1,225000,192515,...,0,0,0,0,0.0,0.0,0.0,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
24995,124995,456247,0,Cash loans,F,N,Y,0,112500,345510,...,0,0,0,0,0.0,0.0,0.0,1.0,0.0,2.0
24996,124996,456248,0,Cash loans,F,N,Y,0,153000,331920,...,0,0,0,0,0.0,0.0,0.0,0.0,0.0,1.0
24997,124997,456251,0,Cash loans,M,N,N,0,157500,254700,...,0,0,0,0,0.0,0.0,0.0,0.0,0.0,1.0
24998,124998,456254,1,Cash loans,F,N,Y,0,171000,370107,...,0,0,0,0,0.0,0.0,0.0,0.0,0.0,0.0


## save as gzip & push to s3

In [6]:
str_filename = 'df_training.gzip'
str_local_path = f'{str_dirname_output}/{str_filename}'
df_training_clean.to_parquet(str_local_path,
             compression='gzip')

NameError: name 'df_training_clean' is not defined

In [39]:
%%time

upload_to_s3(
    aws_access_key_id=AWS_ACCESS_KEY_ID,
    aws_secret_access_key=AWS_SECRET_ACCESS_KEY,
    str_local_path=str_local_path,
    str_bucket_key=f'09_aws_batch/01_create_image/{str_filename}',
    str_bucket_name='20240509-christian-internship',
)

CPU times: total: 62.5 ms
Wall time: 666 ms


In [40]:
str_filename = 'df_validation.gzip'
str_local_path = f'{str_dirname_output}/{str_filename}'
df_validation_clean.to_parquet(str_local_path,
             compression='gzip')

In [41]:
%%time

upload_to_s3(
    aws_access_key_id=AWS_ACCESS_KEY_ID,
    aws_secret_access_key=AWS_SECRET_ACCESS_KEY,
    str_local_path=str_local_path,
    str_bucket_key=f'09_aws_batch/01_create_image/{str_filename}',
    str_bucket_name='20240509-christian-internship',
)

CPU times: total: 109 ms
Wall time: 21.5 s


In [42]:
str_filename = 'df_testing.gzip'
str_local_path = f'{str_dirname_output}/{str_filename}'
df_testing_clean.to_parquet(str_local_path,
             compression='gzip')

In [43]:
%%time

upload_to_s3(
    aws_access_key_id=AWS_ACCESS_KEY_ID,
    aws_secret_access_key=AWS_SECRET_ACCESS_KEY,
    str_local_path=str_local_path,
    str_bucket_key=f'09_aws_batch/01_create_image/{str_filename}',
    str_bucket_name='20240509-christian-internship',
)

CPU times: total: 93.8 ms
Wall time: 368 ms
